# 🚀 High-Performance Code Generator V2

This improved version adds a multi-stage agentic pipeline:
1.  **The Architect**: Transpiles Python to high-performance C++/Rust.
2.  **The Documenter**: Enhances the code with senior-level docstrings and comments.
3.  **The Strategist**: Explains the optimizations used.

**New Features:**
- **Streaming**: Real-time generation of code and documentation.
- **Local Support**: Seamlessly switch between OpenAI and local Ollama models.
- **Tabbed Interface**: Organized view for code, logs, and performance strategy.

In [ ]:
import os
import subprocess
from openai import OpenAI
import gradio as gr
from dotenv import load_dotenv

load_dotenv(override=True)

def get_client(provider="OpenAI"):
    if provider == "OpenAI":
        return OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    else: # Ollama / Local
        return OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [ ]:
ARCHITECT_PROMPT = """You are a specialized Performance Engineer. 
Your task is to convert Python code into the MOST performance-optimized version of the target language.
Use low-level optimizations (SIMD, memory alignment, efficient loops) where applicable.
Return ONLY the code block. No explanations here."""

DOCUMENTER_PROMPT = """You are a Staff Software Engineer.
Given a block of code, add comprehensive, industry-standard documentation:
1. Doxygen/RustDoc style header for functions.
2. High-level explanation of memory management.
Return the code with documentation added. Maintain original logic exactly."""

STRATEGIST_PROMPT = """You are an Algorithm Expert.
Explain exactly why the generated code is faster than Python.
Highlight specific data structures and compiler-level optimizations used."""

In [ ]:
def run_agent_stream(prompt, content, provider, model):
    client = get_client(provider)
    stream = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": content}
        ],
        stream=True
    )
    text = ""
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            text += delta
            yield text

def clean_code(text, lang):
    tag = lang.lower()
    return text.replace(f"```{tag}", "").replace("```", "").strip()

def process_pipeline(python_code, lang, provider, model):
    # Stage 1: Architeching
    yield "Architecting performance code...", "", ""
    generated_raw = ""
    for code in run_agent_stream(ARCHITECT_PROMPT + f" Targeting {lang}.", python_code, provider, model):
        generated_raw = code
        yield "Architecting...", clean_code(generated_raw, lang), ""
    
    # Stage 2: Documenting
    clean_gen = clean_code(generated_raw, lang)
    yield "Adding professional documentation...", clean_gen, ""
    doc_code = ""
    for d_code in run_agent_stream(DOCUMENTER_PROMPT, clean_gen, provider, model):
        doc_code = d_code
        yield "Documenting...", clean_code(doc_code, lang), ""
    
    # Stage 3: Strategy Explanation
    yield "Analyzing performance gains...", clean_code(doc_code, lang), ""
    strategy_text = ""
    for strategy in run_agent_stream(STRATEGIST_PROMPT, doc_code, provider, model):
        strategy_text = strategy
        yield "Ready", clean_code(doc_code, lang), strategy_text

In [ ]:
with gr.Blocks(theme="soft") as demo:
    gr.Markdown("# 🚀 High-Performance Code Generator V2")
    
    with gr.Row():
        with gr.Column(scale=1):
            py_input = gr.Code(label="Python Source", language="python", lines=15)
            lang_select = gr.Dropdown(["C++", "Rust"], label="Target Language", value="C++")
            provider_select = gr.Radio(["OpenAI", "Local (Ollama)"], label="Provider", value="OpenAI")
            model_input = gr.Textbox(label="Model", value="gpt-4o-mini")
            submit_btn = gr.Button("Generate & Document", variant="primary")
            status_label = gr.Label(value="Waiting for input...")
            
        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.TabItem("Generated Code"):
                    code_output = gr.Code(label="Optimized Code", lines=25)
                with gr.TabItem("Performance Analysis"):
                    strategy_output = gr.Markdown()
    
    submit_btn.click(
        process_pipeline,
        inputs=[py_input, lang_select, provider_select, model_input],
        outputs=[status_label, code_output, strategy_output]
    )

demo.launch()